# .bt1 Boundary Training Data — Visual Guide

This notebook loads a `.bt1` file and visualizes the boundary training data used to train the NN model.

A `.bt1` file stores per-iteration **(cfv, cfreach)** pairs at each turn boundary node.
- **cfv** = counterfactual value (what the turn/river subtree returns)
- **cfreach** = opponent's reach probability at the boundary (the NN input)

See `docs/bt1_format.md` for the full format specification.

In [ ]:
import struct
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120

## 1. Load .bt1 File

In [ ]:
BT1_PATH = "../data/bt1/KcQh7s.bt1"
BOUNDARIES_PATH = "../data/bt1/KcQh7s_boundaries.json"
HANDS_PATH = "../data/bt1/KcQh7s_hands.json"

def load_bt1(path):
    """Parse .bt1 binary file."""
    with open(path, "rb") as f:
        magic = f.read(8)
        assert magic == b"BT1\0\0\0\0\0", f"Bad magic: {magic}"

        version = struct.unpack("<I", f.read(4))[0]
        num_oop = struct.unpack("<I", f.read(4))[0]
        num_ip = struct.unpack("<I", f.read(4))[0]
        num_boundaries = struct.unpack("<I", f.read(4))[0]
        num_iterations = struct.unpack("<I", f.read(4))[0]
        starting_pot = struct.unpack("<f", f.read(4))[0]

        num_hands = [num_oop, num_ip]
        records = []
        exploitabilities = []

        for _ in range(num_iterations):
            iteration = struct.unpack("<I", f.read(4))[0]
            exploitability = struct.unpack("<f", f.read(4))[0]
            _reserved = struct.unpack("<I", f.read(4))[0]
            exploitabilities.append(exploitability)

            boundaries = []
            for _ in range(num_boundaries):
                player_data = []
                for player in range(2):
                    opponent = player ^ 1
                    cfv = np.frombuffer(f.read(num_hands[player] * 4), dtype=np.float32).copy()
                    cfreach = np.frombuffer(f.read(num_hands[opponent] * 4), dtype=np.float32).copy()
                    player_data.append((cfv, cfreach))
                boundaries.append(player_data)
            records.append(boundaries)

    return {
        "version": version,
        "num_oop": num_oop,
        "num_ip": num_ip,
        "num_boundaries": num_boundaries,
        "num_iterations": num_iterations,
        "starting_pot": starting_pot,
        "records": records,
        "exploitabilities": np.array(exploitabilities),
    }

bt1 = load_bt1(BT1_PATH)

import os
file_size = os.path.getsize(BT1_PATH)

print(f"File: {BT1_PATH} ({file_size / 1024 / 1024:.1f} MB)")
print(f"Board: KcQh7s")
print(f"OOP hands: {bt1['num_oop']}, IP hands: {bt1['num_ip']}")
print(f"Boundaries: {bt1['num_boundaries']}")
print(f"Iterations: {bt1['num_iterations']}")
print(f"Starting pot: {bt1['starting_pot']}")
print(f"Total training samples: {bt1['num_iterations']} × {bt1['num_boundaries']} × 2 = {bt1['num_iterations'] * bt1['num_boundaries'] * 2:,}")

## 2. Boundary Info

Each boundary represents a different flop action sequence that reaches the turn.
Boundaries are in DFS order of the game tree.

In [ ]:
with open(BOUNDARIES_PATH) as f:
    boundary_info = json.load(f)

print(f"{'Idx':>3}  {'Pot':>5}  Path")
print("-" * 50)
for i, b in enumerate(boundary_info["boundaries"]):
    print(f"{i:3d}  {b['pot']:5.0f}  {b['path']}")

## 3. Exploitability Convergence

How the solver converges over iterations. Exploitability measures how far strategies are from Nash equilibrium.

In [ ]:
exploit_pct = bt1["exploitabilities"] / bt1["starting_pot"] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(exploit_pct)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Exploitability (% of pot)")
ax1.set_title("Exploitability Convergence")
ax1.grid(alpha=0.3)

ax2.plot(exploit_pct)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Exploitability (% of pot)")
ax2.set_title("Exploitability (log scale)")
ax2.set_yscale("log")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial: {exploit_pct[0]:.1f}% of pot")
print(f"Final:   {exploit_pct[-1]:.2f}% of pot")

## 4. CFV Evolution Over Iterations

How CFV values at a boundary change as the solver iterates.
Early iterations are noisy (strategies are unstable), later iterations converge.

In [ ]:
# Track mean CFV per boundary across iterations (OOP player)
n_iter = bt1["num_iterations"]
n_bound = bt1["num_boundaries"]

# Pick 6 boundaries to show
show_boundaries = [0, 1, 5, 12, 18, 24]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, b in enumerate(show_boundaries):
    ax = axes[idx]
    # Track a few specific hands
    hand_indices = [0, bt1["num_oop"] // 4, bt1["num_oop"] // 2, bt1["num_oop"] - 1]
    for h in hand_indices:
        cfvs = [bt1["records"][t][b][0][0][h] for t in range(n_iter)]
        ax.plot(cfvs, alpha=0.7, linewidth=0.8, label=f"hand {h}")
    path = boundary_info["boundaries"][b]["path"]
    pot = boundary_info["boundaries"][b]["pot"]
    ax.set_title(f"B{b}: {path} (pot={pot})", fontsize=9)
    ax.set_xlabel("Iteration", fontsize=8)
    ax.set_ylabel("CFV (chips)", fontsize=8)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7)
fig.suptitle("OOP CFV Evolution per Boundary (selected hands)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. cfreach Evolution Over Iterations

cfreach is the opponent's reach probability — the NN input.
It also changes every iteration as strategies evolve.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, b in enumerate(show_boundaries):
    ax = axes[idx]
    # OOP player's cfreach = IP's reach (length num_ip)
    hand_indices = [0, bt1["num_ip"] // 4, bt1["num_ip"] // 2, bt1["num_ip"] - 1]
    for h in hand_indices:
        reaches = [bt1["records"][t][b][0][1][h] for t in range(n_iter)]
        ax.plot(reaches, alpha=0.7, linewidth=0.8, label=f"opp hand {h}")
    path = boundary_info["boundaries"][b]["path"]
    pot = boundary_info["boundaries"][b]["pot"]
    ax.set_title(f"B{b}: {path} (pot={pot})", fontsize=9)
    ax.set_xlabel("Iteration", fontsize=8)
    ax.set_ylabel("cfreach", fontsize=8)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=7)
fig.suptitle("OOP's cfreach (IP reach) Evolution per Boundary (selected hands)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. CFV Distribution at Final Iteration

Histogram of converged CFV values across all hands, for a few boundaries.

In [ ]:
last = n_iter - 1

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, b in enumerate(show_boundaries):
    ax = axes[idx]
    oop_cfv = bt1["records"][last][b][0][0]
    ip_cfv = bt1["records"][last][b][1][0]

    ax.hist(oop_cfv, bins=50, alpha=0.6, label=f"OOP ({len(oop_cfv)})", density=True)
    ax.hist(ip_cfv, bins=50, alpha=0.6, label=f"IP ({len(ip_cfv)})", density=True)

    path = boundary_info["boundaries"][b]["path"]
    pot = boundary_info["boundaries"][b]["pot"]
    ax.set_title(f"B{b}: {path} (pot={pot})", fontsize=9)
    ax.set_xlabel("CFV (chips)", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

fig.suptitle(f"CFV Distribution at Final Iteration (iter {last})", fontsize=12)
plt.tight_layout()
plt.show()

## 7. cfreach Distribution at Final Iteration

What do the opponent reach probabilities look like when the solver has converged?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, b in enumerate(show_boundaries):
    ax = axes[idx]
    # Player 0 (OOP): cfreach is IP's reach (length num_ip)
    # Player 1 (IP): cfreach is OOP's reach (length num_oop)
    oop_cfreach = bt1["records"][last][b][0][1]  # IP's reach when traversing for OOP
    ip_cfreach = bt1["records"][last][b][1][1]   # OOP's reach when traversing for IP

    ax.hist(oop_cfreach, bins=50, alpha=0.6, label=f"IP reach ({len(oop_cfreach)})", density=True)
    ax.hist(ip_cfreach, bins=50, alpha=0.6, label=f"OOP reach ({len(ip_cfreach)})", density=True)

    path = boundary_info["boundaries"][b]["path"]
    pot = boundary_info["boundaries"][b]["pot"]
    ax.set_title(f"B{b}: {path} (pot={pot})", fontsize=9)
    ax.set_xlabel("Reach probability", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

fig.suptitle(f"cfreach Distribution at Final Iteration (iter {last})", fontsize=12)
plt.tight_layout()
plt.show()

## 8. CFV Heatmap — All Boundaries × All Hands

Shows the full CFV landscape at the final iteration. Each row is a boundary, each column is a hand.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# OOP heatmap
oop_matrix = np.array([bt1["records"][last][b][0][0] for b in range(n_bound)])
im1 = ax1.imshow(oop_matrix, aspect="auto", cmap="RdBu_r", interpolation="nearest")
ax1.set_title(f"OOP CFV ({bt1['num_oop']} hands × {n_bound} boundaries)")
ax1.set_xlabel("Hand index")
ax1.set_ylabel("Boundary index")
plt.colorbar(im1, ax=ax1, label="CFV (chips)")

# IP heatmap
ip_matrix = np.array([bt1["records"][last][b][1][0] for b in range(n_bound)])
im2 = ax2.imshow(ip_matrix, aspect="auto", cmap="RdBu_r", interpolation="nearest")
ax2.set_title(f"IP CFV ({bt1['num_ip']} hands × {n_bound} boundaries)")
ax2.set_xlabel("Hand index")
ax2.set_ylabel("Boundary index")
plt.colorbar(im2, ax=ax2, label="CFV (chips)")

fig.suptitle(f"CFV Heatmap at Final Iteration (iter {last})", fontsize=13)
plt.tight_layout()
plt.show()

## 9. CFV Magnitude by Boundary

Which boundaries have the largest CFV values? Larger pots typically produce larger CFVs.

In [ ]:
oop_mean_abs = [np.abs(bt1["records"][last][b][0][0]).mean() for b in range(n_bound)]
ip_mean_abs = [np.abs(bt1["records"][last][b][1][0]).mean() for b in range(n_bound)]
pots = [boundary_info["boundaries"][b]["pot"] for b in range(n_bound)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(n_bound)
width = 0.35
ax1.bar(x - width/2, oop_mean_abs, width, label="OOP", alpha=0.8)
ax1.bar(x + width/2, ip_mean_abs, width, label="IP", alpha=0.8)
ax1.set_xlabel("Boundary index")
ax1.set_ylabel("Mean |CFV| (chips)")
ax1.set_title("Mean Absolute CFV per Boundary")
ax1.legend()
ax1.grid(alpha=0.3, axis="y")

ax2.scatter(pots, oop_mean_abs, label="OOP", alpha=0.7)
ax2.scatter(pots, ip_mean_abs, label="IP", alpha=0.7)
ax2.set_xlabel("Pot size (chips)")
ax2.set_ylabel("Mean |CFV| (chips)")
ax2.set_title("CFV Magnitude vs Pot Size")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. cfreach vs CFV Relationship

Does the opponent's reach at a boundary predict the CFV? This is what the NN needs to learn.

In [ ]:
# Pick a few (boundary, iteration) pairs to scatter
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

sample_iters = [0, n_iter // 4, n_iter // 2, 3 * n_iter // 4, n_iter - 1]
b = 1  # boundary with moderate pot

for idx, t in enumerate(sample_iters):
    ax = axes[idx]
    cfv = bt1["records"][t][b][0][0]       # OOP CFV
    cfreach = bt1["records"][t][b][0][1]    # IP reach (opponent)

    # cfreach has num_ip entries, cfv has num_oop entries — can't scatter directly
    # Instead show both distributions side by side
    ax.scatter(range(len(cfv)), cfv, s=1, alpha=0.5, label="CFV")
    ax.set_title(f"Iter {t}", fontsize=9)
    ax.set_xlabel("Hand index", fontsize=8)
    ax.set_ylabel("CFV (chips)", fontsize=8)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

# Use last subplot for cfreach
ax = axes[5]
for t in [0, n_iter - 1]:
    cfreach = bt1["records"][t][b][0][1]
    ax.scatter(range(len(cfreach)), cfreach, s=1, alpha=0.5, label=f"cfreach iter {t}")
ax.set_title("IP cfreach at B1", fontsize=9)
ax.set_xlabel("Opponent hand index", fontsize=8)
ax.set_ylabel("Reach probability", fontsize=8)
ax.legend(fontsize=7, markerscale=5)
ax.grid(alpha=0.3)
ax.tick_params(labelsize=7)

path = boundary_info["boundaries"][b]["path"]
fig.suptitle(f"OOP CFV across iterations at B{b} ({path}), and cfreach comparison", fontsize=12)
plt.tight_layout()
plt.show()

## 11. Training Data Summary

What the NN sees: input and output dimensions, valid positions, and target scale.

In [ ]:
num_oop = bt1["num_oop"]
num_ip = bt1["num_ip"]
max_hands = max(num_oop, num_ip)
n_bound = bt1["num_boundaries"]
n_iter = bt1["num_iterations"]
n_samples = n_iter * n_bound * 2

# Count valid output positions
oop_samples = n_iter * n_bound  # one per (iter, boundary)
ip_samples = n_iter * n_bound
valid_outputs = oop_samples * num_oop + ip_samples * num_ip
total_outputs = n_samples * max_hands

# Compute target scale (same as training script)
all_cfvs = []
for t in range(n_iter):
    for b in range(n_bound):
        for p in range(2):
            all_cfvs.append(bt1["records"][t][b][p][0])
all_cfvs = np.concatenate(all_cfvs)
y_scale = np.abs(all_cfvs).max()

print("=== NN Training Data Summary ===")
print(f"Total samples:     {n_samples:,} ({n_iter} iters × {n_bound} boundaries × 2 players)")
print(f"Input dim:         {n_bound + 1 + max_hands} (boundary_oh={n_bound} + player=1 + cfreach={max_hands})")
print(f"Output dim:        {max_hands} (padded to max_hands)")
print(f"Valid outputs:     {valid_outputs:,} / {total_outputs:,} ({100*valid_outputs/total_outputs:.1f}%)")
print(f"  OOP valid:       {num_oop} per sample ({oop_samples:,} samples)")
print(f"  IP valid:        {num_ip} per sample ({ip_samples:,} samples)")
print(f"Target scale:      {y_scale:.4f} chips")
print(f"CFV range:         [{all_cfvs.min():.4f}, {all_cfvs.max():.4f}] chips")
print(f"CFV mean:          {all_cfvs.mean():.6f} chips")
print(f"CFV std:           {all_cfvs.std():.4f} chips")